# Face Recognition Backend Pipeline

This notebook implements the backend-only hackathon pipeline inside `backend/`:

1. read enrollment images from `known_people/<person_name>/`
2. extract face embeddings with `face_recognition`
3. save them to `data/known_faces.json`
4. open the webcam with OpenCV
5. detect faces, match them against the JSON database, and draw `name` or `Unknown`

Recommended folder structure:

```text
backend/
├── playGround.ipynb
├── data/
│   └── known_faces.json
└── known_people/
    ├── Ahmed/
    │   ├── 1.jpg
    │   └── 2.jpg
    └── Sara/
        └── 1.jpg
```


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import cv2
import face_recognition
import numpy as np


def resolve_backend_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / "playGround.ipynb").exists():
        return current
    if (current / "backend" / "playGround.ipynb").exists():
        return current / "backend"
    return current


BACKEND_DIR = resolve_backend_dir()
KNOWN_PEOPLE_DIR = BACKEND_DIR / "known_people"
DATA_DIR = BACKEND_DIR / "data"
DATABASE_PATH = DATA_DIR / "known_faces.json"

SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DEFAULT_TOLERANCE = 0.50
DEFAULT_DETECTION_MODEL = "hog"
DEFAULT_FRAME_SCALE = 0.50

DATA_DIR.mkdir(parents=True, exist_ok=True)
KNOWN_PEOPLE_DIR.mkdir(parents=True, exist_ok=True)
if not DATABASE_PATH.exists():
    DATABASE_PATH.write_text("[]\n", encoding="utf-8")

print(f"Backend directory: {BACKEND_DIR}")
print(f"Known people directory: {KNOWN_PEOPLE_DIR}")
print(f"Database path: {DATABASE_PATH}")
print("If face_recognition is missing, install it with: pip install face_recognition opencv-python numpy")


## Stage A: Enrollment and Database Creation

This stage builds the recognition database from `known_people/`.

Rules used here:

- each folder name is the person identity
- each image should contain exactly one face
- images with zero or multiple faces are skipped
- multiple embeddings are stored per person for robustness


In [ ]:
def list_image_files(folder: Path) -> list[Path]:
    return sorted(
        path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )


def load_face_database(database_path: Path = DATABASE_PATH) -> list[dict[str, Any]]:
    if not database_path.exists():
        return []

    raw_text = database_path.read_text(encoding="utf-8").strip()
    if not raw_text:
        return []

    payload = json.loads(raw_text)
    if isinstance(payload, dict) and "people" in payload:
        return payload["people"]
    if isinstance(payload, list):
        return payload
    raise ValueError("Database must be a list of people or a {'people': [...]} object.")


def save_face_database(database: list[dict[str, Any]], database_path: Path = DATABASE_PATH) -> None:
    database_path.parent.mkdir(parents=True, exist_ok=True)
    database_path.write_text(json.dumps(database, indent=2), encoding="utf-8")


def extract_single_face_encoding(
    image: np.ndarray,
    detection_model: str = DEFAULT_DETECTION_MODEL,
) -> tuple[np.ndarray | None, str | None]:
    locations = face_recognition.face_locations(image, model=detection_model)

    if len(locations) == 0:
        return None, "no_face"
    if len(locations) > 1:
        return None, "multiple_faces"

    encodings = face_recognition.face_encodings(image, known_face_locations=locations)
    if len(encodings) != 1:
        return None, "encoding_failed"

    return encodings[0].astype(np.float32), None


def build_face_database(
    known_people_dir: Path = KNOWN_PEOPLE_DIR,
    database_path: Path = DATABASE_PATH,
    detection_model: str = DEFAULT_DETECTION_MODEL,
) -> tuple[list[dict[str, Any]], list[dict[str, int | str]]]:
    database: list[dict[str, Any]] = []
    report: list[dict[str, int | str]] = []

    if not known_people_dir.exists():
        raise FileNotFoundError(f"Known people directory does not exist: {known_people_dir}")

    for person_dir in sorted(path for path in known_people_dir.iterdir() if path.is_dir()):
        person_name = person_dir.name
        embeddings: list[list[float]] = []
        stats = {
            "person": person_name,
            "accepted": 0,
            "skipped_no_face": 0,
            "skipped_multiple_faces": 0,
            "skipped_encoding_failed": 0,
        }

        for image_path in list_image_files(person_dir):
            image = face_recognition.load_image_file(str(image_path))
            encoding, error = extract_single_face_encoding(
                image,
                detection_model=detection_model,
            )

            if error == "no_face":
                stats["skipped_no_face"] += 1
                continue
            if error == "multiple_faces":
                stats["skipped_multiple_faces"] += 1
                continue
            if error is not None:
                stats["skipped_encoding_failed"] += 1
                continue

            embeddings.append(encoding.tolist())
            stats["accepted"] += 1

        if embeddings:
            database.append({"name": person_name, "embeddings": embeddings})
        report.append(stats)

    save_face_database(database, database_path=database_path)
    return database, report


def summarize_database(database: list[dict[str, Any]]) -> None:
    total_people = len(database)
    total_embeddings = sum(len(person.get("embeddings", [])) for person in database)
    print(f"People enrolled: {total_people}")
    print(f"Total embeddings: {total_embeddings}")
    for person in database:
        print(f"- {person['name']}: {len(person.get('embeddings', []))} embeddings")


def print_enrollment_report(report: list[dict[str, int | str]]) -> None:
    for item in report:
        print(
            f"{item['person']}: accepted={item['accepted']}, "
            f"skipped_no_face={item['skipped_no_face']}, "
            f"skipped_multiple_faces={item['skipped_multiple_faces']}, "
            f"skipped_encoding_failed={item['skipped_encoding_failed']}"
        )


## Stage B: Recognition Helpers

This stage loads the JSON database, flattens it into embeddings, matches each detected face to the nearest known embedding, and labels it as either a person name or `Unknown`.


In [ ]:
def flatten_database(database: list[dict[str, Any]]) -> tuple[list[str], list[np.ndarray]]:
    known_names: list[str] = []
    known_embeddings: list[np.ndarray] = []

    for person in database:
        name = person["name"]
        for embedding in person.get("embeddings", []):
            known_names.append(name)
            known_embeddings.append(np.asarray(embedding, dtype=np.float32))

    return known_names, known_embeddings


def match_face_embedding(
    query_embedding: np.ndarray,
    known_names: list[str],
    known_embeddings: list[np.ndarray],
    tolerance: float = DEFAULT_TOLERANCE,
) -> dict[str, Any]:
    if not known_embeddings:
        return {
            "name": "Unknown",
            "distance": None,
            "confidence": 0.0,
        }

    distances = face_recognition.face_distance(known_embeddings, query_embedding)
    best_index = int(np.argmin(distances))
    best_distance = float(distances[best_index])

    if best_distance < tolerance:
        return {
            "name": known_names[best_index],
            "distance": best_distance,
            "confidence": max(0.0, 1.0 - best_distance),
        }

    return {
        "name": "Unknown",
        "distance": best_distance,
        "confidence": max(0.0, 1.0 - best_distance),
    }


def recognize_faces_in_frame(
    frame: np.ndarray,
    known_names: list[str],
    known_embeddings: list[np.ndarray],
    tolerance: float = DEFAULT_TOLERANCE,
    frame_scale: float = DEFAULT_FRAME_SCALE,
    detection_model: str = DEFAULT_DETECTION_MODEL,
) -> list[dict[str, Any]]:
    if frame_scale <= 0 or frame_scale > 1:
        raise ValueError("frame_scale must be in the range (0, 1].")

    working_frame = frame
    scale_back = 1.0
    if frame_scale != 1.0:
        working_frame = cv2.resize(frame, (0, 0), fx=frame_scale, fy=frame_scale)
        scale_back = 1.0 / frame_scale

    rgb_frame = cv2.cvtColor(working_frame, cv2.COLOR_BGR2RGB)
    locations = face_recognition.face_locations(rgb_frame, model=detection_model)
    encodings = face_recognition.face_encodings(rgb_frame, known_face_locations=locations)

    results: list[dict[str, Any]] = []
    for (top, right, bottom, left), query_embedding in zip(locations, encodings):
        match = match_face_embedding(
            query_embedding,
            known_names,
            known_embeddings,
            tolerance=tolerance,
        )

        results.append(
            {
                "name": match["name"],
                "distance": match["distance"],
                "confidence": match["confidence"],
                "box": (
                    int(left * scale_back),
                    int(top * scale_back),
                    int((right - left) * scale_back),
                    int((bottom - top) * scale_back),
                ),
            }
        )

    return results


def draw_recognition_results(frame: np.ndarray, results: list[dict[str, Any]]) -> np.ndarray:
    for result in results:
        x, y, w, h = result["box"]
        name = result["name"]
        confidence = result["confidence"]

        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        label = name if name == "Unknown" else f"{name} ({confidence:.0%})"

        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        (text_w, text_h), baseline = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            2,
        )
        label_top = max(y - text_h - baseline - 8, 0)
        label_bottom = label_top + text_h + baseline + 8
        cv2.rectangle(frame, (x, label_top), (x + text_w + 10, label_bottom), color, -1)
        cv2.putText(
            frame,
            label,
            (x + 5, label_bottom - baseline - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
        )

    return frame


## Stage C: Live Camera Recognition

This stage opens the webcam, performs live recognition, and overlays bounding boxes and labels.

Press `q` or `Esc` to stop the camera window.


In [ ]:
def run_live_recognition(
    database_path: Path = DATABASE_PATH,
    camera_id: int = 0,
    tolerance: float = DEFAULT_TOLERANCE,
    frame_scale: float = DEFAULT_FRAME_SCALE,
    detection_model: str = DEFAULT_DETECTION_MODEL,
    process_every_n_frames: int = 1,
) -> None:
    database = load_face_database(database_path)
    known_names, known_embeddings = flatten_database(database)

    if not known_embeddings:
        print("Database is empty. Build the database first or every face will show as Unknown.")

    capture = cv2.VideoCapture(camera_id)
    if not capture.isOpened():
        raise RuntimeError(f"Cannot open camera {camera_id}.")

    frame_index = 0
    cached_results: list[dict[str, Any]] = []
    process_every_n_frames = max(1, process_every_n_frames)

    print("Live recognition started. Press 'q' or Esc to quit.")

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                print("Failed to read a frame from the camera.")
                break

            if frame_index % process_every_n_frames == 0:
                cached_results = recognize_faces_in_frame(
                    frame,
                    known_names,
                    known_embeddings,
                    tolerance=tolerance,
                    frame_scale=frame_scale,
                    detection_model=detection_model,
                )

            draw_recognition_results(frame, cached_results)
            cv2.imshow("Face Recognition", frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                break

            frame_index += 1
    finally:
        capture.release()
        cv2.destroyAllWindows()


## Usage

Run the cells top to bottom, then use the examples below.


In [ ]:
# 1. Put enrollment images inside backend/known_people/<person_name>/*.jpg
#    Example:
#    backend/known_people/Ahmed/1.jpg
#    backend/known_people/Ahmed/2.jpg
#    backend/known_people/Sara/1.jpg

# 2. Build or refresh the embeddings database.
# database, report = build_face_database()
# print_enrollment_report(report)
# summarize_database(database)

# 3. Start live webcam recognition.
# run_live_recognition(
#     camera_id=0,
#     tolerance=0.50,
#     frame_scale=0.50,
#     process_every_n_frames=1,
# )

# 4. Optional threshold tuning ideas for your own data:
#    0.45 -> stricter, more Unknown results
#    0.50 -> good default for hackathon demos
#    0.55 or 0.60 -> more permissive, higher false-match risk

database = load_face_database()
summarize_database(database)
